In [25]:
import os
from ultralytics import YOLO
import cv2
from pathlib import Path

# Paths
input_folder = r'C:\Machine_Learning\Scripts\A2\frame'
annotated_folder = os.path.join(input_folder, 'annotated')
labels_folder = os.path.join(input_folder, 'labels')
os.makedirs(annotated_folder, exist_ok=True)
os.makedirs(labels_folder, exist_ok=True)

# Load pretrained YOLOv8 model (nano version for speed)
model = YOLO('yolov8n.pt')

# Process all images (including subfolders)
image_extensions = ['.jpg', '.jpeg', '.png']
image_paths = [p for p in Path(input_folder).rglob('*') if p.suffix.lower() in image_extensions]

for img_path in image_paths:
    results = model(img_path)
    boxes = results[0].boxes
    img = cv2.imread(str(img_path))
    height, width = img.shape[:2]

    label_lines = []
    people_detected = 0
    for box in boxes:
        cls_id = int(box.cls[0])
        if cls_id != 0:  # Only keep 'person' class (ID 0 in COCO)
            continue
        people_detected += 1
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        # Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Convert to YOLO format: class x_center y_center width height (all normalized)
        x_center = (x1 + x2) / 2 / width
        y_center = (y1 + y2) / 2 / height
        box_width = (x2 - x1) / width
        box_height = (y2 - y1) / height
        label_lines.append(f"0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}")

    print(f"{img_path.name} -> People detected: {people_detected}")

    # Save annotated image even if no people were detected (for verification)
    out_img_path = os.path.join(annotated_folder, img_path.name)
    cv2.imwrite(out_img_path, img)

    # Save YOLO annotation
    label_path = os.path.join(labels_folder, img_path.stem + '.txt')
    with open(label_path, 'w') as f:
        f.write('\n'.join(label_lines))

print("✅ Done! Annotated images and YOLO labels saved.")


image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0000.jpg: 384x640 14 persons, 83.2ms
Speed: 3.2ms preprocess, 83.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
frame_0000.jpg -> People detected: 14

image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0001.jpg: 384x640 15 persons, 67.5ms
Speed: 2.4ms preprocess, 67.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
frame_0001.jpg -> People detected: 15

image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0002.jpg: 384x640 14 persons, 66.8ms
Speed: 2.8ms preprocess, 66.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
frame_0002.jpg -> People detected: 14

image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0003.jpg: 384x640 14 persons, 63.4ms
Speed: 2.7ms preprocess, 63.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
frame_0003.jpg -> People detected: 14

image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0004.jp

In [26]:
import os
import shutil
import random

# Paths
base_path = r'C:\Machine_Learning\Scripts\A2\frame'
images_path = os.path.join(base_path, "annotated")
labels_path = os.path.join(base_path, "labels")
dataset_path = os.path.join(base_path, "yolo_dataset")

# Create folders
for folder in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(folder, exist_ok=True)

# Output folders
train_img_dir = os.path.join(dataset_path, "images", "train")
val_img_dir = os.path.join(dataset_path, "images", "val")
train_lbl_dir = os.path.join(dataset_path, "labels", "train")
val_lbl_dir = os.path.join(dataset_path, "labels", "val")

# Get list of images
all_images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.png'))]
random.shuffle(all_images)

split_idx = int(0.8 * len(all_images))
train_files = all_images[:split_idx]
val_files = all_images[split_idx:]

def copy_files(image_list, split):
    img_out_dir = train_img_dir if split == 'train' else val_img_dir
    lbl_out_dir = train_lbl_dir if split == 'train' else val_lbl_dir
    for img_file in image_list:
        lbl_file = os.path.splitext(img_file)[0] + ".txt"
        src_img = os.path.join(images_path, img_file)
        src_lbl = os.path.join(labels_path, lbl_file)
        dst_img = os.path.join(img_out_dir, img_file)
        dst_lbl = os.path.join(lbl_out_dir, lbl_file)
        if os.path.exists(src_img) and os.path.exists(src_lbl):
            shutil.copy2(src_img, dst_img)
            shutil.copy2(src_lbl, dst_lbl)

copy_files(train_files, 'train')
copy_files(val_files, 'val')

print(f"✅ Done. Copied {len(train_files)} training images and {len(val_files)} validation images.")


✅ Done. Copied 1732 training images and 433 validation images.


In [27]:
from ultralytics import YOLO
import os

# Base paths
base_path = r'C:\Machine_Learning\Scripts\A2\frame'
dataset_path = os.path.join(base_path, "yolo_dataset")
yaml_path = os.path.join(base_path, "dataset.yaml")

# Create dataset.yaml
with open(yaml_path, "w") as f:
    f.write(f"""
path: {dataset_path}
train: images/train
val: images/val
nc: 1
names: ['person']
""")

# Load and train YOLOv8
model = YOLO("yolov8n.pt")

model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
    batch=8,
    patience=20,
    project=os.path.join(base_path, "runs"),
    name="person_detector",
    exist_ok=True
)

# Evaluate trained model
metrics = model.val()
print("Validation metrics:", metrics)

# Inference on validation images
results = model.predict(
    source=os.path.join(dataset_path, 'images', 'val'),
    save=True,
    conf=0.3
)

print("✅ Done! Check predictions in the 'runs' folder.")


Ultralytics 8.3.107  Python-3.13.3 torch-2.6.0+cpu CPU (AMD Ryzen 7 7435HS)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=C:\Machine_Learning\Scripts\A2\frame\dataset.yaml, epochs=20, time=None, patience=20, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=C:\Machine_Learning\Scripts\A2\frame\runs, name=person_detector, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, sh

train: Scanning C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\train... 1732 images, 0 backgrounds, 0 corrupt


train: New cache created: C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\train.cache


val: Scanning C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\val... 433 images, 0 backgrounds, 0 corrupt: 100


val: New cache created: C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\val.cache
Plotting labels to C:\Machine_Learning\Scripts\A2\frame\runs\person_detector\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to C:\Machine_Learning\Scripts\A2\frame\runs\person_detector
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...


20 epochs completed in 2.188 hours.
Optimizer stripped from C:\Machine_Learning\Scripts\A2\frame\runs\person_detector\weights\last.pt, 6.2MB
Optimizer stripped from C:\Machine_Learning\Scripts\A2\frame\runs\person_detector\weights\best.pt, 6.2MB

Validating C:\Machine_Learning\Scripts\A2\frame\runs\person_detector\weights\best.pt...
Ultralytics 8.3.107  Python-3.13.3 torch-2.6.0+cpu CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:21


                   all        433       7473      0.998      0.987      0.995      0.992
Speed: 0.9ms preprocess, 41.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Machine_Learning\Scripts\A2\frame\runs\person_detector
Ultralytics 8.3.107  Python-3.13.3 torch-2.6.0+cpu CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\val.cache... 433 images, 0 backgrounds, 0 corrup
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 55/55 [00:20


                   all        433       7473      0.998      0.987      0.995      0.992
Speed: 0.6ms preprocess, 39.3ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Machine_Learning\Scripts\A2\frame\runs\person_detector
Validation metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F48CB04670>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.0280

In [28]:
from ultralytics import YOLO


model = YOLO(r"C:\Machine_Learning\Scripts\A2\frame\runs\person_detector\weights\best.pt")
metrics = model.val()

print(metrics)

Ultralytics 8.3.107  Python-3.13.3 torch-2.6.0+cpu CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning C:\Machine_Learning\Scripts\A2\frame\yolo_dataset\labels\val.cache... 433 images, 0 backgrounds, 0 corrup
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:21


                   all        433       7473      0.998      0.987      0.995      0.992
Speed: 0.6ms preprocess, 40.6ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs\detect\val5
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F458566F90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.

In [35]:
results = model(r"C:\Machine_Learning\Scripts\A2\frame\frames\frame_0014.jpg")
results[0].show()


image 1/1 C:\Machine_Learning\Scripts\A2\frame\frames\frame_0014.jpg: 384x640 (no detections), 51.4ms
Speed: 1.4ms preprocess, 51.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
